0️⃣  Check individual samples
1️⃣  **Compare replicates**
2️⃣  Compare materials

# 1️⃣ Compare Replicates - OER Analysis (HZB CE-NESD group)

The NESD group performs replicate measurements for each experimental condition. In this notebook, these replicates can be linked together and selected which of the replicate measurements should be included in the analysis. Please note that currently only measurements in the same NOMAD upload can be compared.

The calculations and desired plots are based on information provided by Bastian Schmiedecke.

In [1]:
%%capture
%matplotlib widget
#!pip install requests_cache

import matplotlib.pyplot as plt
import time
import requests
import pandas as pd
import numpy as np
from datetime import datetime

import plotly.graph_objects as go
import plotly.io as pio
import json

from scipy.signal import find_peaks

import ipywidgets as widgets
from IPython.display import display

import sys
sys.path.insert(1, '../python-scripts-c6fxKDJrSsWp1xCxON1Y7g')
sys.path.insert(1, '../../python-scripts-c6fxKDJrSsWp1xCxON1Y7g')
from api_calls import *
from plotly_helpers import *

url = "https://nomad-hzb-ce.de/nomad-oasis/api/v1"

import os
token = os.environ['NOMAD_CLIENT_ACCESS_TOKEN']

In [2]:
# all ipywidgets
uploads = get_all_uploads(url, token, number_of_uploads=200)
upload_names = [u.get("upload_name","--no-name--") for u in uploads]

upload_selector = widgets.Dropdown(
    options=['-- please select an upload --'] + upload_names,
    value='-- please select an upload --',
    description='NOMAD upload:',
    style={'description_width': 'initial'}
)

folder_selector = widgets.Dropdown(
    description="Select Folder:",
    style={'description_width': 'initial'}
)

multi_selector_entries = widgets.SelectMultiple(
    description='Select the samples you would like to compare:',
    layout=widgets.Layout(width='80%'),
    style={'description_width': 'initial'}
)

show_sample_details_button = widgets.Button(
    description="Show plots for selected samples",
    button_style='info',
    layout=widgets.Layout(width='auto')
)

save_comparison_button = widgets.Button(
    description="Write the selected analysis to NOMAD",
    button_style='danger',
    layout=widgets.Layout(width='auto')
)

overview_output = widgets.Output()
multi_select_output = widgets.Output()
folder_select_output = widgets.Output()
analysis_output = widgets.Output()
nomad_prepare_output = widgets.Output()
nomad_write_output = widgets.Output()

oer_compare_json = {}

out1 = widgets.Output()
out2 = widgets.Output()
out3 = widgets.Output()
out4 = widgets.Output()
out5 = widgets.Output()

tab = widgets.Tab(children=[out1, out2, out3, out4, out5])
tab.set_title(0, 'Charge Density')
tab.set_title(1, 'Overpotential')
tab.set_title(2, 'EIS')
tab.set_title(3, 'Tafel Slopes')
tab.set_title(4, 'ECSA (electrochemical active surface area)')

In [3]:
def reset_nomad_write_button():
    nomad_prepare_output.clear_output()
    nomad_write_output.clear_output()
    with nomad_prepare_output:
        print('If there is no button:')
        print('Please validate your data with the "Show plots for selected samples" button before writing to NOMAD.') 

def on_upload_change(change):
    if change['type'] == 'change' and change['name'] == 'value':
        overview_output.clear_output()
        folder_select_output.clear_output()
        analysis_output.clear_output()
        reset_nomad_write_button()
        with overview_output:
            upload_name = change['new']
            analysis_entries = get_specific_entrytype_of_uploads(url, token, [upload_name], 'CE_NESD_OERAnalysis', with_meta=True)
            analysis_entries.sort(key=lambda entry: entry[0].get('name', ''))
            entry_names = [entry[0].get('name') for entry in analysis_entries]
            folder_names = sorted({s.rsplit("/", 1)[0] for s in entry_names})
            print(f'Number of OER entries: {len(analysis_entries)}')
        if len(entry_names) != 0:
            options = [
                (entry[0].get('name', ''), entry)
                for entry in analysis_entries
            ]
            multi_selector_entries.options = options
        if len(folder_names) != 0:
            folder_selector.options = folder_names
            with folder_select_output:
                folder_select_output.clear_output()
                display(folder_selector)

        if upload_selector.value == '-- please select an upload --':
            with folder_select_output:
                print('Please select an upload before inspecting individual runs.')

upload_selector.observe(on_upload_change)

display(upload_selector, overview_output)

Dropdown(description='NOMAD upload:', options=('-- please select an upload --', 'NESD Showcase OER Analysis - …

Output()

In [4]:
def on_select_change(change):
    with multi_select_output:
        reset_nomad_write_button()
        analysis_output.clear_output()
        multi_select_output.clear_output()
        selected_entries = change['new']
        for e in selected_entries:
            entry = e[0]
            name = entry['name']
            if entry.get('outputs') is None:
                print('⚠️ Please reprocess the selected upload before selecting this sample.')
            else:
                print(f'{name}: charge_density = {entry['outputs'][0]['charge_density']}, overpotential = {entry['outputs'][0]['overpotential_at_10mA_cm2']}')

multi_selector_entries.observe(on_select_change, names='value')

display(multi_selector_entries, multi_select_output)

SelectMultiple(description='Select the samples you would like to compare:', layout=Layout(width='80%'), option…

Output()

In [5]:
def plot_mean_with_spread(title, x_label, y_label, y_values):
    mean_val = np.mean(y_values)
    min_val = np.min(y_values)
    max_val = np.max(y_values)

    bar = go.Bar(
        x=[x_label],
        y=[mean_val],
        name='Mean',
        marker_color='lightblue'
    )

    line = go.Scatter(
        x=[x_label, x_label],
        y=[min_val, max_val],
        mode='lines+markers',
        line=dict(color='black', width=2),
        name='Range (min-max)',
        marker=dict(symbol='circle', size=8)
    )

    fig = go.Figure(data=[bar, line])

    fig.update_layout(
        title=title,
        #xaxis_title=x_label,
        yaxis_title=y_label,
        #yaxis=dict(range=[0, max(max_val*1.1, mean_val*1.2)]),
        template='plotly_white'
    )
    return fig

def get_plot_from_entry(entry, index):
    plot_dict = entry.get('figures')[index].get('figure')
    fig = pio.from_json(pio.to_json(plot_dict))
    return fig
    
def get_charge_density_plot_from_entry(entry):
    return get_plot_from_entry(entry, 0)

def get_overpotential_plot_from_entry(entry):
    return get_plot_from_entry(entry, 1)

def get_eis_plot_from_entry(entry):
    return get_plot_from_entry(entry, 2)

def get_tafel_plot_from_entry(entry):
    return get_plot_from_entry(entry, 3)

In [6]:
def get_reference(upload_id, entry_id):
    return f'../uploads/{upload_id}/archive/{entry_id}#/data'

def get_comparison_dict(oer_analysis_list, quantity_name):
    comp_dict = {}
    overview_list = [entry[0]['outputs'][0][quantity_name] for entry in oer_analysis_list]
    mean = np.mean(overview_list)
    closest_entry = min(
        oer_analysis_list,
        key=lambda e: abs(e[0]['outputs'][0][quantity_name] - mean)
    )
    comp_dict[f'{quantity_name}_list'] = overview_list
    comp_dict[f'mean_{quantity_name}'] = mean
    comp_dict['closest_entry'] = closest_entry[0]
    comp_dict[f'selected_electrode_{quantity_name}'] = {'name': closest_entry[1].get('entry_name'), 'reference': get_reference(closest_entry[1].get('upload_id'), closest_entry[1].get('entry_id'))}
    return comp_dict

def get_sample_list(oer_analysis_list):
    samples = []
    for analysis_entry in oer_analysis_list:
        oer_data, oer_metadata = analysis_entry
        samples += oer_data.get('outputs')[0].get('samples')
    return samples

def get_input_ref_list(oer_analysis_list):
    inputs = []
    for analysis_entry in oer_analysis_list:
        oer_data, oer_metadata = analysis_entry
        inputs.append({'name': oer_metadata.get('entry_name'), 'reference': get_reference(oer_metadata.get('upload_id'), oer_metadata.get('entry_id'))})
    return inputs

def get_nomad_entry_json(charge_density_dict, overpotential_dict, oer_analysis_list):
    json_data = {}
    json_data['data'] = {}
    json_data['data']['m_def'] = 'nomad_chemical_energy.schema_packages.ce_nesd_package.CE_NESD_OERCompareReplicates'
    oer_compare_json = {
        'charge_densities': charge_density_dict.get('charge_density_list'),
        'overpotentials': overpotential_dict.get('overpotential_at_10mA_cm2_list'),
        'reaction_type': 'OER',
        'samples': get_sample_list(oer_analysis_list),
        'selected_electrode_charge_density': charge_density_dict.get('selected_electrode_charge_density'),
        'selected_electrode_overpotential': overpotential_dict.get('selected_electrode_overpotential_at_10mA_cm2'),
    }
    json_data['data']['inputs'] = get_input_ref_list(oer_analysis_list)
    json_data['data']['outputs'] = [oer_compare_json]
    return json_data

def add_name_to_json(json_dict, file_path):
    if isinstance(json_dict.get('data'), dict):
        json_dict['data']['name'] = f'{file_path}oer_analysis_compare_replicates'
        outputs = json_dict['data'].get('outputs')
        if isinstance(outputs, list) and len(outputs) > 0 and isinstance(outputs[0], dict):
            outputs[0]['name'] = f'{file_path}oer_analysis_compare_replicas'

def prepare_nomad_writing(oer_analysis_list, nomad_json):
    with nomad_prepare_output:
        nomad_prepare_output.clear_output()
        common_prefix = os.path.commonprefix([entry[0]['name'] for entry in oer_analysis_list])
        print(f'Upload name: {upload_selector.value}')
        print(f'Folder path: {common_prefix}')
        add_name_to_json(nomad_json, common_prefix)
        display(save_comparison_button)
        
def on_show_button_click(b):
    global oer_compare_json
    with analysis_output:
        analysis_output.clear_output()
        selected_entries = list(multi_selector_entries.value)
        x_label = 'material'

        if not selected_entries:
            print('⚠️ Please select at least one entry.')
            return
        with out1:
            out1.clear_output()
            cd_dict = get_comparison_dict(selected_entries, 'charge_density')
            print(f'Mean of selected charge_densities: {cd_dict.get('mean_charge_density'):.3f}mC/cm²')
            fig1 = plot_mean_with_spread('Charge densities', x_label, 'Charge Density [mC/cm²]', cd_dict.get('charge_density_list'))
            fig1.show()
            print(f'Entry closest to mean: {cd_dict.get('closest_entry')['name']} (charge_density = {cd_dict.get('closest_entry')['outputs'][0]['charge_density']})')
            fig2 = get_charge_density_plot_from_entry(cd_dict.get('closest_entry')['outputs'][0])
            fig2.show()
        with out2:
            out2.clear_output()
            op_dict = get_comparison_dict(selected_entries, 'overpotential_at_10mA_cm2')
            print(f'Mean of selected overpotentials: {op_dict.get('mean_overpotential_at_10mA_cm2'):.3f}mV')
            fig1 = plot_mean_with_spread('Overpotentials at 10 mA/cm²', x_label, 'Overpotential [mV]', op_dict.get('overpotential_at_10mA_cm2_list'))
            fig1.show()
            print(f'Entry closest to mean: {op_dict.get('closest_entry')['name']} (overpotential = {op_dict.get('closest_entry')['outputs'][0]['overpotential_at_10mA_cm2']})')
            fig2 = get_overpotential_plot_from_entry(op_dict.get('closest_entry')['outputs'][0])
            fig2.show()
        with out3:
            out3.clear_output()
            print('EIS for selected electrode that is closest to overpotential mean.')
            fig = get_eis_plot_from_entry(op_dict.get('closest_entry')['outputs'][0])
            fig.show()
        with out4:
            out4.clear_output()
            print('Tafel Slope for selected electrode that is closest to overpotential mean.')
            fig = get_tafel_plot_from_entry(op_dict.get('closest_entry')['outputs'][0])
            fig.show()
        with analysis_output:
            analysis_output.clear_output()
            display(tab)
            oer_compare_json = get_nomad_entry_json(cd_dict, op_dict, selected_entries)
            prepare_nomad_writing(selected_entries, oer_compare_json)

show_sample_details_button.on_click(on_show_button_click)
display(show_sample_details_button, analysis_output)


Button(button_style='info', description='Show plots for selected samples', layout=Layout(width='auto'), style=…

Output()

### Write selected OER Comparison to NOMAD

The following button creates a new NOMAD entry. **Please note that clicking the button is not easily reversible.**

If you already have an analysis entry at the given folder path it will be overwritten.

In [7]:
display(nomad_prepare_output)

Output()

In [8]:
def write_new_entry_to_nomad(url, token, upload_id, folder_path, json_dict):
    complete_url = f'{url}/uploads/{upload_id}/raw/{folder_path}'
    params = {
        "overwrite_if_exists": "true",
        "wait_for_processing": "false",
        "include_archive": "false",
        "auto_decompress": "true",
    }
    headers = {
        'accept': 'application/json',
        'Authorization': f'Bearer {token}',
    }
    files = {
        "file": ("oer_analysis_compare_replicates.archive.json", json.dumps(json_dict).encode("utf-8"), "application/json")
    }
    response = requests.put(complete_url, headers=headers, params=params, files=files)
    res = response.json()
    return res

def on_write_to_nomad_button_click(b):
    with nomad_write_output:
        nomad_write_output.clear_output()
        upload_id = multi_selector_entries.value[0][1].get('upload_id')
        file_path = os.path.commonprefix([entry[0]['name'] for entry in multi_selector_entries.value])
        response = write_new_entry_to_nomad(url, token, upload_id, file_path, oer_compare_json)
        if response.get('data', {}).get('last_status_message') != 'Process completed successfully':
            print(response)
        else:
            print('✅ Entry written to NOMAD')
            print(f'Visit preview: https://nomad-hzb-ce.de/nomad-oasis/gui/user/uploads/upload/id/{upload_id}/files/{file_path.replace('%', '!3').replace(' ', '%20')}oer_analysis_compare_replicates.archive.json/archive/data')


save_comparison_button.on_click(on_write_to_nomad_button_click)
display(nomad_write_output)


Output()

In [9]:
#with open(f'oer_compare.archive.json', 'w') as f:
#    json.dump(oer_compare_json, f, indent=2)